# Seasonal Adjustment (Monthly, UK Calendar) — Module & Diagnostics
This notebook demonstrates the seasonal adjustment utility in `seasonal_adjustment.py`
with monthly data, UK-specific working-day calendar (bank holidays), trading-day and Easter effects,
and an optional X-13 path for closer alignment with JDemetra+.


In [ ]:
import pandas as pd
import numpy as np
from seasonal_adjustment import seasonal_adjust, plot_sa

# Optional: if you intend to use X-13ARIMA-SEATS, ensure statsmodels can find the X-13 binary.
# In statsmodels, this typically works out of the box if the binary is on PATH.
# Otherwise, you can configure the path using statsmodels.tsa.x13.x13_arima_analysis arguments.


## Load your monthly data
Provide a monthly pandas Series `y` indexed by a fixed monthly DatetimeIndex (e.g., end-of-month).
Replace the synthetic example below with your actual data (CSV/Excel read).


In [ ]:
# Example: synthetic monthly series with multiplicative seasonality
idx = pd.date_range('2015-01-31', periods=132, freq='M')
rng = np.random.default_rng(42)
season = np.tile([0.90, 0.92, 0.95, 0.98, 1.02, 1.05, 1.08, 1.07, 1.03, 1.00, 0.97, 0.93], 11)
trend = 100 * (1 + 0.003) ** np.arange(len(idx))
irreg = rng.normal(0, 0.8, len(idx))
y = pd.Series(trend * season * np.exp(irreg/50), index=idx, name='y')
y.head()


## Run seasonal adjustment (UK working days, trading-day, length-of-month, Easter)
Set `method='auto'` to prefer X-13 when available; otherwise STL fallback.


In [ ]:
res = seasonal_adjust(
    y,
    method='auto',            # try X-13, else STL
    multiplicative=True,
    use_trading_day=True,
    use_length_effect=True,
    use_easter=True,
    easter_k=8,
    use_uk_working_days=True,
    outlier_method='hampel'
)
res.diagnostics


In [ ]:
_ = plot_sa(y, res, title='Seasonal Adjustment (UK calendar; auto method)')


## Components
Access the adjusted series and components below.


In [ ]:
sa = res.sa
seasonal = res.seasonal
trend = res.trend
irregular = res.irregular
sa.head(), seasonal.head()


### Notes on matching JDemetra+ more closely
- Install and let statsmodels find the **X-13ARIMA-SEATS** binary, then run with `method='x13'`.
- If your series uses country-specific holiday exceptions (e.g., special jubilees), add them via
  the `extra_uk_holidays` parameter.
- For diagnostics, `res.diagnostics['auto_arima_summary']` (if `pmdarima` is installed) shows the
  identified ARIMA model on the pre-adjusted working series.
